## Noteobook to analyse and create a Pan Atlas

In this notebook, we will learn how to load the different data files and identify doublets (cells that were mistakenly captured together instead of individually).

In [ ]:
import os, sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    REPO = "/content/test_course"
    if not os.path.exists(REPO):
        !git clone --quiet --depth 1 https://github.com/antparleo/test_course.git {REPO}
    %cd {REPO}
    !pip install -q -r scripts/requirements.txt
    main_dir = Path(REPO)
else:
    main_dir = Path("/lustre/scratch125/cellgen/vento/ap58/PanAtlas")
    os.chdir(main_dir)

results_dir = main_dir / "results" / "00-preprocessing"
results_dir.mkdir(parents=True, exist_ok=True)

print(f"Colab: {IN_COLAB} | working dir: {os.getcwd()}")

hello


## Libraries

Below are the libraries (they would be like tools) that we will use in this analysis.
They must be installed in your conda environment beforehand so that the notebook can run without errors.

In [3]:
import pandas as pd
import scanpy as sc
import scrublet as scr
import numpy as np
import matplotlib.pyplot as plt
from scripts.utils.panatlas_utils import score_cell_status, cc_scoring, bh, bonf # This is a own library, you don't need install you just use it.
from scipy.sparse import csr_matrix
import anndata as ad
import pickle as pkl
import scipy
%matplotlib inline # Sometimes, plots do not show, this can help you.

In [5]:
sc.set_figure_params(dpi=80) # Low quality of images to fast analysis.

## Read data

## Metadata from reproductivecellatalas.com

We use the HECA metadata because it is part of this study.

The total number of cells in the metadata may not match the number in the raw data. This is expected, as some cells are removed during quality control.

If you are analysing new data, you will need to provide your own metadata and identify the cell types during the analysis process.

In [6]:
adata_reprocell = ad.io.read_h5ad('data/endometriumAtlasV2_cells_with_counts.h5ad')
adata_reprocell.obs

,n_genes,sample,library,Processing,Treatment,10x kit,percent_mito,n_counts,scrublet_score,genotype,...,Stage,phase,dataset,Biopsy_type,Tissue_sampled,Age,Endometrial_pathology,celltype,lineage,label_long
HCA_A_RepT_RNA13247830_AAACCTGAGCGATGAC-Mareckova,1981,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.036783,3956.0,0.033156,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,ePV_1a,Mesenchymal,19 | ePV 1a
HCA_A_RepT_RNA13247830_AAACCTGAGCTAGGCA-Mareckova,3492,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.042249,9546.0,0.029188,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,eStromal,Mesenchymal,23 | eStromal
HCA_A_RepT_RNA13247830_AAACCTGCACCAACCG-Mareckova,2967,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.092998,7188.0,0.075581,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,Venous,Endothelial,32 | Venous
HCA_A_RepT_RNA13247830_AAACCTGCACCTGGTG-Mareckova,2597,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.035448,6029.0,0.069252,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,mPV,Mesenchymal,18 | mPV
HCA_A_RepT_RNA13247830_AAACCTGCATAGAAAC-Mareckova,3319,HCA_A_RepT_RNA13247830,HCA_A_RepT_RNA13247830,Fresh,Coll.+Trypsin,3' v3.1,0.036459,8571.0,0.054963,A70,...,Proliferative,G1,Mareckova,Whole_Uterus,eutopic_endometrium,37,C,Venous,Endothelial,32 | Venous
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
GSM5572240_TTTGGTTCATCGATGT-Lai,3876,GSM5572240,GSM5572240,Fresh,Unknown,nan,0.105244,19302.0,0.142857,GSM5572240,...,Secretory Early-Mid,G1,Lai,Superficial_Eutopic,eutopic_endometrium,nan,C,dStromal_late,Mesenchymal,27 | dStromal late
GSM5572240_TTTGGTTGTAGCGCCT-Lai,3662,GSM5572240,GSM5572240,Fresh,Unknown,nan,0.096900,18212.0,0.125628,GSM5572240,...,Secretory Early-Mid,G1,Lai,Superficial_Eutopic,eutopic_endometrium,nan,C,dStromal_mid,Mesenchymal,26 | dStromal mid
GSM5572240_TTTGGTTTCCGTTTCG-Lai,4347,GSM5572240,GSM5572240,Fresh,Unknown,nan,0.079747,23705.0,0.094972,GSM5572240,...,Secretory Early-Mid,G1,Lai,Superficial_Eutopic,eutopic_endometrium,nan,C,dStromal_mid,Mesenchymal,26 | dStromal mid
GSM5572240_TTTGGTTTCTGCTTAT-Lai,3661,GSM5572240,GSM5572240,Fresh,Unknown,nan,0.095066,16930.0,0.149338,GSM5572240,...,Secretory Early-Mid,G1,Lai,Superficial_Eutopic,eutopic_endometrium,nan,C,dStromal_mid,Mesenchymal,26 | dStromal mid


## Read the data

We load all the files from the /data/' folder.
However, for this analysis, we only keep the samples that are listed in the HECA metadata.

In [9]:
samples = os.listdir(main_dir+'/data/endometrium_Wang2020')

In [10]:
metadata = adata_reprocell.obs.query('sample in @samples')

In [11]:
metadata.columns

Index(['n_genes', 'sample', 'library', 'Processing', 'Treatment', '10x kit',
       'percent_mito', 'n_counts', 'scrublet_score', 'genotype',
       'Library_genotype', 'Group', 'Endometriosis_stage',
       'Hormonal treatment', 'Binary Stage', 'Stage', 'phase', 'dataset',
       'Biopsy_type', 'Tissue_sampled', 'Age', 'Endometrial_pathology',
       'celltype', 'lineage', 'label_long'],
      dtype='str')

In this step, we select only the columns that are relevant for our analysis.
You should choose the columns that are useful for your specific research question.

If you are working with previously annotated data, make sure to keep the cell annotation columns, as they contain important information about cell types or cell identities.

In [12]:
metadata_wang = metadata.loc[:,['sample','library','Processing','Treatment','10x kit','genotype','Library_genotype','Stage','phase','Biopsy_type','Group','Endometriosis_stage','Hormonal treatment', 'Binary Stage','Biopsy_type', 'Tissue_sampled', 'Age', 'Endometrial_pathology','celltype', 'lineage', 'label_long']]

In [13]:
# Samples included in the study
samples = metadata_wang['sample'].unique().tolist()

## Detection of doublets

Each dataset contains a raw matrix and a filtered matrix.
We will use the filtered matrix because it contains only high-quality cells.
The raw matrix is only used if we need to troubleshoot or check for abnormalities.

Bear in mind, where your data are to put a correct path in the function sc.read_10x_mtx().

During this step, we define three quality control thresholds:
* min_genes (minimum number of genes detected per cell)
* min_cells (minimum number of cells in which a gene must be detected)
* percent_mito (maximum percentage of mitochondrial gene expression per cell)

In [16]:
min_genes = 1000
min_cells = 5
percent_mito = 0.2

In [17]:
adatas ={}

for s in samples:
    
    print(s)

    # Load the different samples
    adata_sample = sc.read_10x_mtx(path=main_dir+"data/endometrium_Wang2020/"+s+"/outs/filtered_feature_bc_matrix")

    # Add donor info to cell barcode
    adata_sample.obs_names = [s +'_'+ i.split('-')[0] for i in adata_sample.obs_names]
    
    # We show number of cells and genes before filtering. 
    print(f'Number of cells: {adata_sample.obs.shape[0]}')
    print(f'Number of genes: {adata_sample.var.shape[0]}\n')

    # Basic filter to improve performance scrublet
    sc.pp.filter_cells(adata_sample, min_genes = min_genes)
    sc.pp.filter_genes(adata_sample, min_cells = min_cells)

    # Look for mitochondrial genes (we convert all to lower to avoid mistakes)
    mito_genes = [name for name in adata_sample.var_names if name.lower().startswith('mt-')]

    # for each cell compute fraction of counts in mito genes vs. all genes
    # the `.A1` is only necessary as X is sparse (to transform to a dense array after summing)
    adata_sample.obs['percent_mito'] = np.sum(
        adata_sample[:, mito_genes].X, axis=1).A1 / np.sum(adata_sample.X, axis=1).A1
    
    # Filter 
    adata_sample = adata_sample[adata_sample.obs['percent_mito'] < percent_mito, :]

    # Run scrublet
    np.random.seed(9)
    scrub = scr.Scrublet(adata_sample.X)
    doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose=0)

    # Add info to anndata
    adata_sample.obs['doublet_scores'] = doublet_scores
    adata_sample.obs['predicted_doublet'] = predicted_doublets

    # Add to list to concatenate
    adatas[s] = adata_sample


GSM4577306
Number of cells: 5134
Number of genes: 36601



/tmp/ipykernel_3443059/1945022744.py:36: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['doublet_scores'] = doublet_scores


GSM4577307
Number of cells: 7673
Number of genes: 36601



/tmp/ipykernel_3443059/1945022744.py:36: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['doublet_scores'] = doublet_scores


GSM4577308
Number of cells: 17798
Number of genes: 36601



...

/tmp/ipykernel_3443059/1945022744.py:36: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['doublet_scores'] = doublet_scores


GSM4577314
Number of cells: 6979
Number of genes: 36601



/tmp/ipykernel_3443059/1945022744.py:36: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['doublet_scores'] = doublet_scores


GSM4577315
Number of cells: 10120
Number of genes: 36601



/tmp/ipykernel_3443059/1945022744.py:36: ImplicitModificationWarning: Trying to modify attribute `.obs` of view, initializing view as actual.
  adata_sample.obs['doublet_scores'] = doublet_scores


### Did you get an error when running Scrublet?

Scrublet applies its own internal filtering steps before calculating doublet scores.

If your sample has low quality or unusual characteristics, these filters may remove too many genes. In some cases, even all of them. When this happens, Scrublet will return an error.

If this is your case, you can uncomment the code below and adapt it to your own dataset to manually adjust the filtering settings.

In [ ]:
# We just take one as an example
# a = sc.read_10x_mtx(path=main_dir+"data/endometrium_Wang2020/"+'SAMN23461118'+"/outs/filtered_feature_bc_matrix")
# sc.pp.filter_genes(a,min_cells=2)
# a.shape # Only 10 cells, if we add the other filters the number of genes will be 0.

Now, we will propagate the doublet scores by running a quick, simplified single-cell analysis workflow.
First, we create a new folder inside the results directory to store the output generated during this step.

In [18]:
scorenames = ['doublet_scores','scrublet_cluster_score','zscore','bh_pval','bonf_pval']
if not os.path.exists(results_dir+'/scrublet_scores'):
    os.mkdir(results_dir+'/scrublet_scores')

In [ ]:
adatas_info = {}

for sample, adata_sample in adatas.items():

#overcluster prep. run turbo basic scanpy pipeline

    # Normalize
    sc.pp.normalize_total(adata_sample)
    sc.pp.log1p(adata_sample)

    # Select features
    sc.pp.highly_variable_genes(adata_sample, min_mean=0.0125, max_mean=3, min_disp=0.5)
    adata_sample = adata_sample[:, adata_sample.var['highly_variable']].copy()

    # Scale the data and perform the first dimensional reduciton (PCA)
    sc.pp.scale(adata_sample, max_value=10)
    sc.tl.pca(adata_sample, svd_solver='arpack')
    sc.pp.neighbors(adata_sample)

    #overclustering proper - do basic clustering first, then cluster each cluster
    sc.tl.leiden(adata_sample)
    adata_sample.obs['leiden'] = [str(i) for i in adata_sample.obs['leiden']]
    
    # For each cluster create a subset and redo leiden and annot to the main one
    for clus in np.unique(adata_sample.obs['leiden']):
        adata_sub = adata_sample[adata_sample.obs['leiden']==clus].copy()
        sc.tl.leiden(adata_sub)
        adata_sub.obs['leiden'] = [clus+','+i for i in adata_sub.obs['leiden']]
        adata_sample.obs.loc[adata_sub.obs_names,'leiden'] = adata_sub.obs['leiden']

    #compute the cluster scores - the median of Scrublet scores per overclustered cluster

    for clus in np.unique(adata_sample.obs['leiden']):
        adata_sample.obs.loc[adata_sample.obs['leiden']==clus, 'scrublet_cluster_score'] = \
            np.median(adata_sample.obs.loc[adata_sample.obs['leiden']==clus, 'doublet_scores'])
    
    #now compute doublet p-values. figure out the median and mad (from above-median values) for the distribution
    med = np.median(adata_sample.obs['scrublet_cluster_score'])
    mask = adata_sample.obs['scrublet_cluster_score']>med
    mad = np.median(adata_sample.obs['scrublet_cluster_score'][mask]-med)

    
    #let's do a one-sided test. the Bertie write-up does not address this but it makes sense
    zscores = (adata_sample.obs['scrublet_cluster_score'].values - med) / (1.4826 * mad)
    adata_sample.obs['zscore'] = zscores
    pvals = 1-scipy.stats.norm.cdf(zscores)
    adata_sample.obs['bh_pval'] = bh(pvals)
    adata_sample.obs['bonf_pval'] = bonf(pvals)

    #create results data frame for single sample and copy stuff over from the adata object
    scrublet_sample = pd.DataFrame(0, index=adata_sample.obs_names, columns=scorenames)
    for score in scorenames:
        scrublet_sample[score] = adata_sample.obs[score]

    #write out complete sample scores
    scrublet_sample.to_csv(results_dir + '/scrublet_scores/'+sample+'.csv')

    adatas_info[sample] = adata_sample

### We save the all the information generated

In [ ]:
with open(results_dir+'/adatas_info.pkl', 'wb') as f:
    pkl.dump(adatas_info, f)

In [20]:
metadata_wang.to_csv(results_dir+'/metadata_wang.csv')